<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/openailike_local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using OpenAILike with a Local Inference Server

Many local inference servers (e.g., [Ollama](https://ollama.com), [vLLM](https://docs.vllm.ai), [LM Studio](https://lmstudio.ai), [Rapid-MLX](https://github.com/raullenchai/Rapid-MLX)) expose an OpenAI-compatible API. The `OpenAILike` class lets you connect LlamaIndex to any of these servers.

Unlike hosted providers, local servers don't advertise model capabilities such as context window size or tool-calling support. `OpenAILike` lets you set these explicitly so LlamaIndex can make the right decisions at query time.

## Installation

In [ ]:
%pip install llama-index-llms-openai-like

## Basic Setup

Point `api_base` at your local server and set the model capabilities explicitly.

In [ ]:
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model="my-local-model",
    api_base="http://localhost:8080/v1",
    api_key="fake-key",  # most local servers don't require a real key
    context_window=4096,  # set to your model's true context size
    is_chat_model=True,
    is_function_calling_model=False,  # set True only if your server supports tool calls
)

## Completion

In [ ]:
response = llm.complete("What is LlamaIndex?")
print(response)

## Chat Messages

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(role="user", content="What is retrieval augmented generation?"),
]

response = llm.chat(messages)
print(response)

## Streaming

In [ ]:
response = llm.stream_complete("Explain RAG in 3 sentences.")
for chunk in response:
    print(chunk.delta, end="", flush=True)

In [ ]:
messages = [
    ChatMessage(role="user", content="What are embeddings?"),
]

response = llm.stream_chat(messages)
for chunk in response:
    print(chunk.delta, end="", flush=True)

## Model Discovery

Many local servers expose an `/v1/models` endpoint that lists available models. You can use this to discover valid model IDs before configuring `OpenAILike`.

In [ ]:
import httpx

API_BASE = "http://localhost:8080/v1"

response = httpx.get(f"{API_BASE}/models")
for model in response.json()["data"]:
    print(model["id"])

## Capability Reference

Local servers don't report model capabilities automatically. Use these parameters to set them explicitly:

| Parameter | Default | When to override |
|-----------|---------|------------------|
| `context_window` | 3900 | **Always** — set to your model's true context size |
| `is_chat_model` | `False` | Set `True` for chat/instruct models (most modern models) |
| `is_function_calling_model` | `False` | Set `True` if your server supports OpenAI-style tool calls |
| `should_use_structured_outputs` | `False` | Set `True` if the server supports `response_format` for structured output |
| `tokenizer` | `None` | Set to a HuggingFace tokenizer name (e.g. `"meta-llama/Llama-3.1-8B"`) for accurate token counting |
| `max_tokens` | `None` | Maximum tokens to generate per request |
| `temperature` | 0.1 | Sampling temperature |
| `timeout` | 60.0 | Seconds to wait before timing out a request |

## Example Configurations for Popular Servers

### vLLM

```bash
# Start server
vllm serve meta-llama/Llama-3.1-8B-Instruct --port 8080
```

In [ ]:
llm_vllm = OpenAILike(
    model="meta-llama/Llama-3.1-8B-Instruct",
    api_base="http://localhost:8080/v1",
    api_key="fake-key",
    context_window=131072,
    is_chat_model=True,
    is_function_calling_model=True,  # vLLM supports tool calls
    tokenizer="meta-llama/Llama-3.1-8B-Instruct",
)

### LM Studio

Start LM Studio, load a model, and enable the local server (default port 1234).

In [ ]:
llm_lmstudio = OpenAILike(
    model="llama-3.1-8b-instruct",  # check LM Studio for the exact model ID
    api_base="http://localhost:1234/v1",
    api_key="lm-studio",
    context_window=131072,
    is_chat_model=True,
    is_function_calling_model=False,
)

### Rapid-MLX (Apple Silicon)

```bash
# Install and start server
pip install rapid-mlx
rapid-mlx serve mlx-community/Llama-3.1-8B-Instruct-4bit --port 8080
```

In [ ]:
llm_rapidmlx = OpenAILike(
    model="mlx-community/Llama-3.1-8B-Instruct-4bit",
    api_base="http://localhost:8080/v1",
    api_key="fake-key",
    context_window=8192,
    is_chat_model=True,
    is_function_calling_model=False,
)